# Stable Diffusion WebUI — полный Google Colab

Ноутбук обновлён для AUTOMATIC1111: проверяет GPU, ставит системные инструменты, получает официальный WebUI и позволяет настроить Drive, расширения и модель. Python/PyTorch вручную не переустанавливаются: установщик WebUI использует зависимости, совместимые с выбранной версией репозитория.

## Запуск с iPhone
1. Откройте **Runtime → Change runtime type → GPU**.
2. В меню **Runtime** выберите **Run all**. При первом запуске подтвердите подключение Google Drive, если оставили его включённым.
3. Дождитесь ссылки `gradio.live` в последней ячейке. Не публикуйте её: любой, у кого есть ссылка, может открыть интерфейс.

Первая установка может занимать время. Google Colab — временная среда; файлы в `/content` удаляются после сброса runtime. Хранение моделей на Drive сохраняет их между запусками, но может замедлить загрузку.

**Важно:** это не гарантирует доступность бесплатного GPU, моделей или сторонних расширений. Если ячейка завершилась ошибкой, остановитесь на ней и пришлите её текст/скриншот — не нужно вручную искать строку в большом ноутбуке.

In [ ]:
#@title 1. Проверка GPU и установка WebUI
import os, sys, subprocess, shutil
from pathlib import Path

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('GPU не найден. В Colab выберите Runtime → Change runtime type → GPU, затем перезапустите эту ячейку.')
print('Python:', sys.version.split()[0])
print('GPU:', gpu.stdout.strip())

USE_GOOGLE_DRIVE = True #@param {type:'boolean'}
UPDATE_EXISTING_WEBUI = False #@param {type:'boolean'}
WEBUI_DIR = '/content/stable-diffusion-webui'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    MODEL_ROOT = '/content/drive/MyDrive/SD-WebUI/models'
else:
    MODEL_ROOT = WEBUI_DIR + '/models'

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','git','aria2','libgl1','libglib2.0-0'], check=True)

if not os.path.isdir(WEBUI_DIR):
    subprocess.run(['git','clone','--depth','1','https://github.com/AUTOMATIC1111/stable-diffusion-webui.git',WEBUI_DIR], check=True)
elif UPDATE_EXISTING_WEBUI:
    # Fast-forward only: never discard local edits or reset the install.
    subprocess.run(['git','-C',WEBUI_DIR,'pull','--ff-only'], check=True)
else:
    print('Используется уже существующая копия WebUI; обновление отключено.')

# Optionally persist model folders without placing the whole application on slow Drive storage.
if USE_GOOGLE_DRIVE:
    model_names = ['Stable-diffusion','VAE','Lora','LyCORIS','hypernetworks','ESRGAN','RealESRGAN','SwinIR','GFPGAN','Codeformer']
    os.makedirs(MODEL_ROOT, exist_ok=True)
    for name in model_names:
        source = os.path.join(WEBUI_DIR,'models',name)
        target = os.path.join(MODEL_ROOT,name)
        os.makedirs(os.path.dirname(source), exist_ok=True)
        os.makedirs(target, exist_ok=True)
        if os.path.islink(source):
            if os.path.realpath(source) != os.path.realpath(target):
                os.unlink(source)
                os.symlink(target,source)
        elif os.path.isdir(source):
            for item in os.listdir(source):
                old = os.path.join(source,item)
                new = os.path.join(target,item)
                if not os.path.exists(new):
                    shutil.move(old,new)
            shutil.rmtree(source)
            os.symlink(target,source)
        else:
            os.symlink(target,source)

print('WebUI:', WEBUI_DIR)
print('Каталог моделей:', MODEL_ROOT)
print('Установщик WebUI при первом запуске загрузит необходимые Python-зависимости. Не запускайте параллельно pip install/обновление torch.')

In [ ]:
#@title 2. Дополнительные расширения (по желанию)
#@markdown Установите нужные расширения до первого запуска WebUI. Можно снять галочки, если важнее максимально простой первый старт.
INSTALL_CONTROLNET = True #@param {type:'boolean'}
INSTALL_ADETAILER = True #@param {type:'boolean'}
INSTALL_DYNAMIC_PROMPTS = True #@param {type:'boolean'}
INSTALL_TAG_COMPLETE = True #@param {type:'boolean'}
INSTALL_IMAGE_BROWSER = False #@param {type:'boolean'}
INSTALL_REGIONAL_PROMPTER = False #@param {type:'boolean'}

exts = []
if INSTALL_CONTROLNET: exts.append(('sd-webui-controlnet','https://github.com/Mikubill/sd-webui-controlnet.git'))
if INSTALL_ADETAILER: exts.append(('adetailer','https://github.com/Bing-su/adetailer.git'))
if INSTALL_DYNAMIC_PROMPTS: exts.append(('sd-dynamic-prompts','https://github.com/adieyal/sd-dynamic-prompts.git'))
if INSTALL_TAG_COMPLETE: exts.append(('a1111-sd-webui-tagcomplete','https://github.com/DominikDoom/a1111-sd-webui-tagcomplete.git'))
if INSTALL_IMAGE_BROWSER: exts.append(('stable-diffusion-webui-images-browser','https://github.com/AlUlkesh/stable-diffusion-webui-images-browser.git'))
if INSTALL_REGIONAL_PROMPTER: exts.append(('sd-webui-regional-prompter','https://github.com/hako-mikan/sd-webui-regional-prompter.git'))

ext_root = os.path.join(WEBUI_DIR,'extensions')
os.makedirs(ext_root,exist_ok=True)
for name,url in exts:
    dest = os.path.join(ext_root,name)
    if os.path.isdir(dest):
        print('Уже установлено:',name)
        continue
    print('Клонирование:',name)
    subprocess.run(['git','clone','--depth','1',url,dest],check=True)
print('Расширений установлено:',len(exts))
print('Примечание: у сторонних расширений свои зависимости; WebUI проверит их при запуске. Совместимость расширений не контролируется этим репозиторием.')

In [ ]:
#@title 3. Скачать модель (опционально)
#@markdown Модель не включена автоматически: файлы занимают гигабайты, а некоторые требуют авторизации/принятия лицензии. Укажите прямую ссылку на файл, к которому у вас есть доступ.
MODEL_URL = '' #@param {type:'string'}
MODEL_FILENAME = '' #@param {type:'string'}

if MODEL_URL.strip():
    filename = MODEL_FILENAME.strip() or MODEL_URL.split('?')[0].rstrip('/').split('/')[-1]
    if not filename or filename in ('.','..'):
        raise ValueError('Укажите имя файла модели в поле MODEL_FILENAME.')
    os.makedirs(MODEL_ROOT + '/Stable-diffusion',exist_ok=True)
    destination = os.path.join(MODEL_ROOT,'Stable-diffusion',filename)
    subprocess.run(['aria2c','--continue=true','--max-connection-per-server=8','--split=8','--dir',os.path.dirname(destination),'--out',filename,MODEL_URL],check=True)
    print('Модель сохранена:',destination)
else:
    print('Пропуск скачивания. Добавьте модель позже в папку:',MODEL_ROOT + '/Stable-diffusion')

In [ ]:
#@title 4. Запустить WebUI
#@markdown `gradio.live` — публичная ссылка. Не отправляйте её посторонним. Начальные параметры ниже ориентированы на распространённые Colab GPU; при нехватке памяти включите LOW_VRAM.
LOW_VRAM = False #@param {type:'boolean'}
MED_VRAM = True #@param {type:'boolean'}
NO_HALF_VAE = True #@param {type:'boolean'}
ENABLE_XFORMERS = False #@param {type:'boolean'}
ENABLE_API = False #@param {type:'boolean'}
EXTRA_ARGS = '' #@param {type:'string'}

args = [sys.executable,'launch.py','--share','--theme','dark']
if LOW_VRAM: args.append('--lowvram')
elif MED_VRAM: args.append('--medvram')
if NO_HALF_VAE: args.append('--no-half-vae')
if ENABLE_XFORMERS: args.append('--xformers')
if ENABLE_API: args.append('--api')
if EXTRA_ARGS.strip(): args.extend(EXTRA_ARGS.split())
print('Запускаю официальный установщик WebUI. Первый запуск скачает дополнительные компоненты и может занять время.')
os.chdir(WEBUI_DIR)
subprocess.run(args,check=True)